# Lab 40: Machine Learning Capstone: Train and Evaluate Your Own Model

**Duration:** 4 hours. **Level:** intermediate. **Work mode:** individual or pairs.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
This is the last lab of the course. You do a complete machine learning project, from the question to a working model that someone else can run. You choose a problem and a metric, prepare the data without leaks, set a **baseline**, compare models, tune the best one fairly, test it **once** on data it has never seen, write the training as a script that gives **the same result every time**, keep **training notes**, and build an **inference demo** that predicts a new example. Everything you learned in Labs 31 to 39 comes together.

**By the end of this lab you can:**
1. Define a machine learning problem with a label, a metric and a baseline, before you train.
2. Prepare, split, train, compare and tune models without leaking the test set.
3. Evaluate the final model once, read the errors and compare the result with the baseline.
4. Make the training reproducible: a script, a fixed seed, saved files and notes.
5. Deliver a model with an inference demo and present the evidence.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Project milestone 7: Train and Evaluate Your Own AI Model.** Submit a working model, the comparison with the baseline, the evaluation on held-out data, the training notes and an inference demo. The check task at the end lists the evidence.

## 2. Background

**The life of a machine learning project.**

| Stage | Question | Lab |
|---|---|---|
| 1. Problem | What do I predict? How do I measure success? What is the dumb answer (baseline)? | this lab |
| 2. Data | Is it clean? Is there a leak? Is the test set locked? | 35 |
| 3. Models | Which model beats the baseline? | 36, 37, 38 |
| 4. Tuning | Which settings are best, chosen without the test set? | 39 |
| 5. Exam | One honest evaluation, then read the errors | 31, 39 |
| 6. Delivery | A script anyone can run again, notes, and a way to use the model | this lab |

**Choosing the metric.** Accuracy is not always the right measure.

| Problem | Worse mistake | Watch |
|---|---|---|
| Is a tumour dangerous? | Missing a dangerous one (false negative) | **Recall** of the dangerous class |
| Is the e-mail spam? | Throwing away a real e-mail (false positive) | **Precision** of the spam class |
| Which digit is it? | All mistakes are similar | **Accuracy** and the confusion matrix |

**Reproducible.** Someone else (or you, next month) must get the **same numbers** from the same data. That needs: a fixed random seed, the same split, the same code in a script (not only in a notebook), the version of the libraries written down, and the model saved to a file.

**Training notes.** A short text next to the model: what data, what split, what model and settings, what scores (including the baseline), when, and with which versions. Without notes, a model file is a mystery.

**Inference.** Using the saved model on a **new** example. It must prepare the example in exactly the same way as in training (the same pipeline), which is why the model file contains the whole pipeline, not only the classifier.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(40)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Define the Project (30 min)

### Step 1: Choose the Problem
Choose **one** of three projects by setting `CHOICE` in the cell below.

| `CHOICE` | Problem | Label | Good metric |
|---|---|---|---|
| `"cancer"` | Is a tumour measurement malignant or benign? (569 patients, 30 measurements) | malignant / benign | **recall** of malignant, then F1 |
| `"digits"` | Which digit is in an 8 x 8 picture? (1797 pictures) | 0 to 9 | accuracy, confusion matrix |
| `"readings"` | Should the fan be on? (your sensor table `readings.csv` of Lab 35) | fan on / off | recall of *fan on*, then F1 |

Everything below works for any of them. Run the cell: it writes a small module `ml_project.py` (the data loader and the model builder) next to the notebook, because the training script and the demo must import the same code.

In [ ]:
%%writefile ml_project.py
"""The problem, the data and the model of the capstone. Used by the notebook, by train.py and by predict.py."""
import os

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

CHOICE = "cancer"          # "cancer", "digits" or "readings"
SEED = 2026


def load_project(choice=CHOICE):
    """Returns X (a table of numbers), y (the labels), the feature names and the class names."""
    if choice == "cancer":
        d = load_breast_cancer()
        return pd.DataFrame(d.data, columns=d.feature_names), pd.Series(d.target), list(d.feature_names), ["malignant", "benign"]
    if choice == "digits":
        d = load_digits()
        names = [f"pixel{i}" for i in range(64)]
        return pd.DataFrame(d.data, columns=names), pd.Series(d.target), names, [str(i) for i in range(10)]
    if choice == "readings":
        path = next((p for p in ("../lab35/readings.csv", "readings.csv") if os.path.exists(p)), None)
        if path is None:
            raise SystemExit("readings.csv not found. Do Lab 35 first, or choose another project.")
        df = pd.read_csv(path).drop_duplicates()
        df["room"] = df.room.str.strip().str.lower().map({"kitchen": 0, "lab": 1, "garage": 2})
        df["hour"] = pd.to_datetime(df.time).dt.hour
        df.loc[(df.temperature < -10) | (df.temperature > 50), "temperature"] = np.nan
        names = ["temperature", "humidity", "hour", "room"]
        return df[names], df["fan"], names, ["fan off", "fan on"]
    raise SystemExit("CHOICE must be cancer, digits or readings")


def split_data(X, y):
    """The same split every time: 60 % train, 20 % validation, 20 % test (the test set is locked until the end)."""
    X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=SEED, stratify=y_rest)
    return X_train, X_val, X_test, y_train, y_val, y_test


def make_model():
    """The final model: the whole pipeline (fill, scale, classify). Change it after you have chosen and tuned in Part C."""
    return Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
                     ("model", RandomForestClassifier(n_estimators=100, random_state=SEED))])

### Step 2: The Project Card
Write your plan **before** you look at any score.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- CHOICE and the problem in one sentence:
- The label: what is 1 and what is 0:
- The metric you will watch and why (which mistake is worse):
- The baseline you must beat:
- One risk (a possible leak, a rare class, a sensor problem):

---
# Part B: Look at the Data and Lock the Test Set (30 min)

### Step 1: Inspect
Load the data, look at its size, the balance of the classes, and the empty cells. Then split it: the test set is locked from now on.

In [ ]:
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ml_project
importlib.reload(ml_project)
from ml_project import CHOICE, SEED, load_project, make_model, split_data

X, y, feature_names, class_names = load_project(CHOICE)
print(f"Project: {CHOICE} | rows {len(X)} | features {X.shape[1]} | classes {class_names}")
print("Share of each class:", {class_names[int(k)]: f"{v * 100:.0f} %" for k, v in y.value_counts(normalize=True).sort_index().items()})
print("Empty cells:", int(X.isna().sum().sum()))

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)
print(f"train {len(X_train)} | validation {len(X_val)} | test {len(X_test)} (locked)")
y.value_counts().sort_index().rename(index=lambda i: class_names[int(i)]).plot.bar(figsize=(4, 2.6))
plt.show()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Rows, features and the share of each class:
- Is the data balanced? Is accuracy a safe metric for it:
- How many empty cells, and what does the pipeline do with them:

---
# Part C: Baseline, Models and Tuning (60 min)

### Step 1: The Baseline
The dumb answer: always the most common class. You must beat it. The cell also defines `report`, which prints accuracy, precision, recall and F1 for the class you care about (`POSITIVE`) on the **validation** set.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

POSITIVE = 0 if CHOICE == "cancer" else 1          # the class you must not miss: malignant (0), fan on (1); change it for digits


def report(model, X_part, y_part, label=""):
    predicted = model.predict(X_part)
    scores = {"accuracy": accuracy_score(y_part, predicted)}
    if CHOICE != "digits":
        scores.update(precision=precision_score(y_part, predicted, pos_label=POSITIVE), recall=recall_score(y_part, predicted, pos_label=POSITIVE),
                      f1=f1_score(y_part, predicted, pos_label=POSITIVE))
    else:
        scores["f1 (macro)"] = f1_score(y_part, predicted, average="macro")
    print(f"{label:<28}" + "  ".join(f"{k} {v * 100:5.1f} %" for k, v in scores.items()))
    return scores


baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
base_scores = report(baseline, X_val, y_val, "baseline (validation)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The baseline scores:
- Which score of the baseline is 0, and what does it tell you about accuracy:

### Step 2: Compare Three Models
Each model is inside a pipeline (fill, scale, model). They are compared with 5-fold cross-validation on the training set: the validation set and the test set are not used to choose.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def pipeline(model):
    return Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", model)])


candidates = {
    "logistic regression": pipeline(LogisticRegression(max_iter=2000)),
    "k nearest neighbours": pipeline(KNeighborsClassifier(n_neighbors=5)),
    "random forest": pipeline(RandomForestClassifier(n_estimators=100, random_state=SEED)),
}
from sklearn.metrics import make_scorer

if CHOICE == "digits":
    scorer, metric_name = "accuracy", "accuracy"
else:
    scorer, metric_name = make_scorer(recall_score, pos_label=POSITIVE), "recall"          # the class you must not miss
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring=scorer)
    print(f"{name:<22} cross-validated {metric_name} {scores.mean() * 100:5.1f} %  spread {scores.std() * 100:4.1f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The three scores and the spread:
- Which model do you choose and why (watch the spread):
- Does the winner beat the baseline by more than the spread:

### Step 3: Tune One Hyperparameter at a Time
Tune the chosen model on the **validation** set. Change one setting, write the result, then the next. The cell shows the pattern for the random forest; change it to your model (`k` for the nearest neighbours, `C` for the logistic regression).

In [ ]:
for depth in (2, 4, 8, None):
    model = pipeline(RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=SEED)).fit(X_train, y_train)
    report(model, X_val, y_val, f"max_depth = {depth}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The hyperparameter you tuned and the values you tried:
- The best value and its validation score:
- The final settings that you put in make_model() of ml_project.py (edit the file, then run Part B Step 1 again):

---
# Part D: The Exam, Once (35 min)

### Step 1: Train the Final Model and Test It
Train `make_model()` on the training set and test it on the **locked test set**. This is the only time the test set is used. Compare with the baseline.

In [ ]:
final = make_model().fit(X_train, y_train)
test_scores = report(final, X_test, y_test, "final model (TEST SET)")
base_test = report(baseline, X_test, y_test, "baseline    (TEST SET)")

### Step 2: Read the Errors
The confusion matrix shows the kind of mistakes. The cell also lists some wrong examples, because the errors teach you what to improve.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

predicted = final.predict(X_test)
ConfusionMatrixDisplay(confusion_matrix(y_test, predicted), display_labels=class_names).plot(cmap="Blues", colorbar=False)
plt.show()
wrong = np.where(predicted != y_test.to_numpy())[0]
print(f"Wrong: {len(wrong)} of {len(y_test)}")
for i in wrong[:5]:
    print(f"  row {X_test.index[i]}: real {class_names[int(y_test.iloc[i])]}, predicted {class_names[int(predicted[i])]}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The test scores of the final model and of the baseline:
- How close are the test scores to the validation scores? What does it tell you about overfitting to the validation set:
- The most important kind of mistake (for example false negatives) and what it would cost in real life:

---
# Part E: Make It Reproducible (45 min)

### Step 1: The Training Script
A notebook is good for exploring. A **script** is good for repeating. `train.py` loads the data with the same function, makes the same split, trains the same model, evaluates it on the test set, and saves the model (`model.joblib`) and the numbers (`metrics.json`) with the versions of the libraries.

In [ ]:
%%writefile train.py
"""Train the capstone model, test it once, and save the model and the numbers."""
import json
import platform
import sys
from datetime import datetime

import joblib
import numpy as np
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

from ml_project import CHOICE, SEED, load_project, make_model, split_data

np.random.seed(SEED)
X, y, feature_names, class_names = load_project(CHOICE)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

model = make_model().fit(X_train, y_train)
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
predicted = model.predict(X_test)

metrics = {
    "project": CHOICE, "seed": SEED, "rows": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "test_accuracy": round(float(accuracy_score(y_test, predicted)), 4),
    "test_f1_macro": round(float(f1_score(y_test, predicted, average="macro")), 4),
    "baseline_accuracy": round(float(accuracy_score(y_test, baseline.predict(X_test))), 4),
    "model": str(model.named_steps["model"]),
    "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "numpy": np.__version__},
}
joblib.dump({"model": model, "features": feature_names, "classes": class_names}, "model.joblib")
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps({k: metrics[k] for k in ("project", "test_accuracy", "test_f1_macro", "baseline_accuracy")}))

### Step 2: Run It Twice
Run the script two times in separate processes. If the project is reproducible, the numbers are identical.

In [ ]:
import json
import subprocess
import sys

results = []
for run in (1, 2):
    done = subprocess.run([sys.executable, "train.py"], capture_output=True, text=True)
    if done.returncode != 0:
        print(done.stderr[-600:])
        break
    results.append(json.load(open("metrics.json")))
    print(f"run {run}: accuracy {results[-1]['test_accuracy']}  f1 {results[-1]['test_f1_macro']}  baseline {results[-1]['baseline_accuracy']}")
print("Identical results:", len(results) == 2 and results[0] == results[1])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The numbers of the two runs:
- Are they identical? What would break the reproducibility (name two things):

### Step 3: The Training Notes
The cell writes `TRAINING_NOTES.md` from the numbers, so the notes can not disagree with the model. Open the file and add your own lines (the data, the choice of the metric, the risks).

In [ ]:
from datetime import datetime

m = json.load(open("metrics.json"))
notes = f"""# Training notes

- Date: {datetime.now():%Y-%m-%d %H:%M}
- Project: {m['project']}   |   seed: {m['seed']}
- Data: rows train {m['rows']['train']}, validation {m['rows']['validation']}, test {m['rows']['test']} (the test set was used once)
- Model: {m['model'][:200]}
- Test accuracy: {m['test_accuracy']}   |   test F1 (macro): {m['test_f1_macro']}
- Baseline (most common class) accuracy: {m['baseline_accuracy']}
- Versions: {m['versions']}

## My notes
- Metric I watched and why:
- Mistakes that matter most:
- What I would try next:
"""
open("TRAINING_NOTES.md", "w").write(notes)
print(notes)

---
# Part F: The Inference Demo (35 min)

### Step 1: Predict a New Example
`predict.py` loads `model.joblib` (the whole pipeline) and predicts. The model needs the features in the same order and units as in training. The cell writes the script, then uses its function on a test example it has not trained on, and measures the time.

In [ ]:
%%writefile predict.py
"""Load the saved model and predict for one new example (a dictionary of feature values)."""
import time

import joblib
import pandas as pd

saved = joblib.load("model.joblib")


def predict(example):
    """example: {feature name: value}. Returns (class name, how sure the model is)."""
    row = pd.DataFrame([example], columns=saved["features"])
    probabilities = saved["model"].predict_proba(row)[0]
    best = int(probabilities.argmax())
    return saved["classes"][best], float(probabilities[best])


if __name__ == "__main__":
    start = time.perf_counter()
    example = {name: 0.0 for name in saved["features"]}
    print(predict(example), f"{(time.perf_counter() - start) * 1000:.1f} ms")

In [ ]:
import importlib
import os
import time

import predict
importlib.reload(predict)

example = X_test.iloc[0].to_dict()
answer, sure = predict.predict(example)
print(f"Model says: {answer} ({sure * 100:.0f} % sure) | the truth: {class_names[int(y_test.iloc[0])]}")

started = time.perf_counter()
for i in range(50):
    predict.predict(X_test.iloc[i].to_dict())
print(f"Time for one prediction: {(time.perf_counter() - started) / 50 * 1000:.1f} ms")
print("Files of the project:", sorted(os.listdir(".")))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The prediction, the confidence and the truth:
- The time for one prediction:
- What happens if you give the model the features in another order or a missing feature (try it):

---
# Part G: Present and Deliver (30 min)

### Step 1: The Evidence
Check your files: `ml_project.py`, `train.py`, `predict.py`, `model.joblib`, `metrics.json`, `TRAINING_NOTES.md` and this notebook. Then give a **5-minute demonstration**:
1. The project card: problem, label, metric, baseline, risk.
2. The data: size, balance, what you cleaned, how the test set was locked.
3. The comparison of the models, and the tuning you did.
4. The test result against the baseline and the errors.
5. `python train.py` run twice with the same result, and the notes.
6. A live prediction with `predict.py`.

### Step 2: Make It Your Own (optional extra)
- Change `POSITIVE` and the metric, and show that another choice leads to another model.
- Try a different project (`CHOICE`) and compare how much work was the same.
- Add a threshold: predict *malignant* when the probability is above 0.3 instead of 0.5, and show the effect on recall and precision.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your final model and the test scores:
- How much better is it than the baseline:
- What you would do next if you had one more week:
- The most important thing you learned in the whole course:

> **Congratulations.** You have gone from a blinking Jetson to an AI system that sees, hears, speaks, reasons and acts, and now to a model that you trained and measured yourself. Keep the habits: a baseline first, a locked test set, one change at a time, and evidence for every claim.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The project card is written before the first score (problem, label, metric, baseline, risk) | |
| 2 | The data is inspected, the classes are counted and the test set is locked | |
| 3 | The baseline is set and three models are compared with cross-validation | |
| 4 | One hyperparameter is tuned on the validation set, one at a time | |
| 5 | The final model is tested once, compared with the baseline, and the errors are read | |
| 6 | `train.py` is run twice with identical results | |
| 7 | `TRAINING_NOTES.md` and `metrics.json` are complete | |
| 8 | `predict.py` predicts a new example and the time is measured | |
| 9 | The 5-minute presentation shows the evidence of the six points | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. Why is the baseline set before the model is trained?:
- 2. How did you choose the metric? Which mistake is worse in your project?:
- 3. How did you keep the test set honest?:
- 4. Why is `train.py` better than only a notebook for delivery?:
- 5. What makes a training reproducible? Name four things.:
- 6. What is in the training notes and who needs them?:
- 7. Why does the model file contain the whole pipeline and not only the classifier?:
- 8. Your model has 99 % accuracy and the baseline has 98 %. Is it good? What do you check next?:
- 9. Name one thing in this project that you would change before using the model in a real hospital or factory.:
- 10. Which lab of the course helped you most in this project, and why?:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `ModuleNotFoundError: ml_project` | The first cell of Part A was not run, or you work in another folder | Run the `%%writefile ml_project.py` cell in this notebook |
| `readings.csv not found` | Lab 35 was not done | Choose `cancer` or `digits`, or do Lab 35 first |
| The results change after you edit `ml_project.py` | The notebook still uses the old module | Run the cell of Part B Step 1 again (it reloads the module) |
| The two runs of `train.py` are different | A random part has no seed | Give every model `random_state=SEED`; do not shuffle without a seed |
| `FileNotFoundError: model.joblib` | `train.py` was not run | Run Part E Step 2 |
| `KeyError` or `ValueError` in `predict` | The example has other feature names or a missing value | Use all the features by their exact names (`saved['features']`) |
| The model is worse than the baseline | A bug, a leak removed too much, or the wrong metric | Check the split, the label and `POSITIVE`; compare accuracy with recall |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(40)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- scikit-learn, Model persistence (joblib): https://scikit-learn.org/stable/model_persistence.html
- Google, Rules of Machine Learning: https://developers.google.com/machine-learning/guides/rules-of-ml
- Model cards for model reporting (Mitchell et al.): https://arxiv.org/abs/1810.03993